<a href="https://colab.research.google.com/github/nishthadighe-bit/Data--Engineering-Practicals/blob/main/Practical_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd
import numpy as np

# =====================================================================
# STEP 1: CREATE MOCK DATA SOURCES (Files & Database)
# =====================================================================

# Source 1: CSV File (Customer Demographics)
csv_data = """customer_id,name,age,city
101,Aarav Sharma,28,Mumbai
102,Ananya Patel,34,Bangalore
103,Rohan Verma,INVALID,Delhi
104,Isha Gupta,22,Pune
105,Vikram Singh,45,Hyderabad
"""
with open("customers.csv", "w") as f:
    f.write(csv_data)

# Source 2: SQLite Database (Sales Transactions)
conn_src = sqlite3.connect("source_sales.db")
cursor = conn_src.cursor()
cursor.execute("DROP TABLE IF EXISTS sales")
cursor.execute("""
    CREATE TABLE sales (
        transaction_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        amount REAL,
        status TEXT
    )
""")
sales_records = [
    (1, 101, 1500.50, 'COMPLETED'),
    (2, 102, 2300.00, 'COMPLETED'),
    (3, 103, -500.00, 'FAILED'),      # Invalid negative amount
    (4, 104, 850.75, 'COMPLETED'),
    (5, 105, 3100.20, 'PENDING')
]
cursor.executemany("INSERT INTO sales VALUES (?,?,?,?)", sales_records)
conn_src.commit()

# =====================================================================
# STEP 2: EXTRACTION PHASE
# =====================================================================
print("--- [EXTRACT] Fetching Data from Multiple Sources ---")

# Extract from CSV
df_customers = pd.read_csv("customers.csv")
print("Customers CSV Extracted:")
print(df_customers)

# Extract from SQLite DB
df_sales = pd.read_sql_query("SELECT * FROM sales", conn_src)
conn_src.close()
print("\nSales Database Extracted:")
print(df_sales)

# =====================================================================
# STEP 3: TRANSFORMATION & DATA VALIDATION PHASE
# =====================================================================
print("\n--- [TRANSFORM] Cleaning & Validating Data ---")

# 1. Join Multi-Source Data
df_merged = pd.merge(df_sales, df_customers, on="customer_id", how="inner")

# 2. Data Type Conversion & Validation
df_merged['age'] = pd.to_numeric(df_merged['age'], errors='coerce')  # Convert invalid text to NaN

# 3. Filtering & Cleaning Rules
# Rule A: Keep only completed transactions
df_transformed = df_merged[df_merged['status'] == 'COMPLETED'].copy()

# Rule B: Remove negative or invalid transaction amounts
df_transformed = df_transformed[df_transformed['amount'] > 0]

# Rule C: Handle missing ages with average age imputation
mean_age = df_transformed['age'].mean()
df_transformed['age'] = df_transformed['age'].fillna(mean_age).astype(int)

# 4. Feature Engineering: Add Spending Category
df_transformed['spend_category'] = np.where(
    df_transformed['amount'] >= 2000, 'High Spender', 'Regular Spender'
)

print("Transformed & Validated Data:")
print(df_transformed[['transaction_id', 'name', 'age', 'city', 'amount', 'spend_category']])

# =====================================================================
# STEP 4: LOADING PHASE (Target Data Warehouse)
# =====================================================================
print("\n--- [LOAD] Inserting Transformed Data into Target Warehouse ---")

# Target Database Connection
conn_target = sqlite3.connect("target_warehouse.db")

# Load transformed data into SQLite Data Warehouse table
df_transformed.to_sql("dim_customer_sales", conn_target, if_exists="replace", index=False)

# Verification Query from Target Database
print("Verifying target table contents from SQLite Data Warehouse:")
df_warehouse = pd.read_sql_query("SELECT * FROM dim_customer_sales", conn_target)
print(df_warehouse)

conn_target.close()
print("\nETL Pipeline Execution Finished Successfully!")

--- [EXTRACT] Fetching Data from Multiple Sources ---
Customers CSV Extracted:
   customer_id          name      age       city
0          101  Aarav Sharma       28     Mumbai
1          102  Ananya Patel       34  Bangalore
2          103   Rohan Verma  INVALID      Delhi
3          104    Isha Gupta       22       Pune
4          105  Vikram Singh       45  Hyderabad

Sales Database Extracted:
   transaction_id  customer_id   amount     status
0               1          101  1500.50  COMPLETED
1               2          102  2300.00  COMPLETED
2               3          103  -500.00     FAILED
3               4          104   850.75  COMPLETED
4               5          105  3100.20    PENDING

--- [TRANSFORM] Cleaning & Validating Data ---
Transformed & Validated Data:
   transaction_id          name  age       city   amount   spend_category
0               1  Aarav Sharma   28     Mumbai  1500.50  Regular Spender
1               2  Ananya Patel   34  Bangalore  2300.00     High Sp